# QLoRA 학습 — 충남대 Q&A 시스템

Colab T4 GPU (15GB VRAM) 에서 실행합니다.

| 항목 | 값 |
|------|-----|
| Base | `Qwen/Qwen2.5-7B-Instruct` (4bit NF4) |
| QLoRA | r=16, alpha=32, dropout=0.05, target=q/k/v/o_proj |
| 데이터 | HF Hub `adoveflash/cnu-qa-system` → `data/qa/train.jsonl` |
| 저장 | Google Drive + HF Hub 이중 백업 |

> **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 전체 실행

## 1. 환경 설정

In [ ]:
%%time
!pip install -q torch transformers accelerate bitsandbytes peft datasets huggingface_hub

In [ ]:
import os
import json
import random
import torch

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

# ── 설정값 (여기만 수정) ──
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_REPO = "adoveflash/cnu-qa-system"
DRIVE_OUTPUT = "/content/drive/MyDrive/cnu_lora_adapter"
LOCAL_OUTPUT = "models/lora_adapter"
CKPT_DIR = f"{DRIVE_OUTPUT}/checkpoints"
MAX_LENGTH = 512
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
EVAL_RATIO = 0.1  # 검증셋 비율

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. HF Hub 로그인 & 데이터 다운로드

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Drive 저장 경로: {DRIVE_OUTPUT}")

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import snapshot_download

TRAIN_PATH = "data/qa/train.jsonl"
if not os.path.exists(TRAIN_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["data/qa/train.jsonl"],
    )

with open(TRAIN_PATH) as f:
    all_data = [json.loads(line) for line in f if line.strip()]

# train / eval 분할
random.seed(SEED)
random.shuffle(all_data)
split_idx = max(1, int(len(all_data) * (1 - EVAL_RATIO)))
train_data = all_data[:split_idx]
eval_data = all_data[split_idx:]

print(f"전체: {len(all_data)}건 → 학습: {len(train_data)}건, 검증: {len(eval_data)}건")

## 3. 모델 & 토크나이저 로드

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[1/2] 토크나이저 로드...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[2/2] 모델 로드 (4bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"모델 로드 완료 — VRAM: {vram_gb:.2f} GB")

## 4. LoRA 설정 & 적용

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. 데이터셋 준비

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "당신은 충남대학교 학내 정보 안내 도우미입니다. "
    "주어진 참고 자료를 바탕으로 정확하게 답변하세요. "
    "참고 자료에 없는 내용은 '확인되지 않은 정보입니다'라고 답하세요."
)


def build_dataset(data_list):
    """QA 리스트를 HF Dataset으로 변환한다."""
    texts = []
    for qa in data_list:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)

    def tokenize_fn(examples):
        tokenized = tokenizer(
            examples["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
        )
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    ds = Dataset.from_dict({"text": texts})
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    return ds


train_dataset = build_dataset(train_data)
eval_dataset = build_dataset(eval_data)
print(f"학습 데이터셋: {len(train_dataset)}건")
print(f"검증 데이터셋: {len(eval_dataset)}건")
print(f"샘플 토큰 길이: {len(train_dataset[0]['input_ids'])}")

## 6. 학습

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback
from huggingface_hub import HfApi


class DriveBackupCallback(TrainerCallback):
    """에폭 종료 시 HF Hub에 백업한다."""
    def on_save(self, args, state, control, **kwargs):
        epoch = int(state.epoch) if state.epoch else 0
        print(f"\n에폭 {epoch} → HF Hub 백업 중...")
        try:
            api = HfApi()
            ckpts = sorted(
                [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
                key=lambda x: int(x.split("-")[1]),
            )
            if ckpts:
                latest_path = os.path.join(CKPT_DIR, ckpts[-1])
                api.upload_folder(
                    folder_path=latest_path,
                    path_in_repo="models/lora_adapter",
                    repo_id=HF_REPO,
                )
                print(f"  백업 완료: {ckpts[-1]}")
        except Exception as e:
            print(f"  백업 실패 (학습은 계속): {e}")


training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=SEED,
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[DriveBackupCallback()],
)

# 체크포인트에서 이어서 학습
resume_ckpt = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume_ckpt = os.path.join(
            CKPT_DIR,
            sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1],
        )
        print(f"체크포인트에서 재개: {resume_ckpt}")

print("학습 시작!")
trainer.train(resume_from_checkpoint=resume_ckpt)
print("학습 완료!")

## 7. 학습 결과 확인

In [ ]:
# 학습 로그 출력
for log in trainer.state.log_history:
    if "eval_loss" in log:
        print(f"에폭 {log.get('epoch', '?'):.0f} — train_loss: {log.get('loss', 'N/A')}, eval_loss: {log['eval_loss']:.4f}")

In [ ]:
# 어댑터 저장 (로컬 + Drive)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
model.save_pretrained(LOCAL_OUTPUT)
tokenizer.save_pretrained(LOCAL_OUTPUT)
print(f"로컬 저장 완료: {LOCAL_OUTPUT}")

# Drive에도 복사
import shutil
drive_adapter = f"{DRIVE_OUTPUT}/final_adapter"
if os.path.exists(drive_adapter):
    shutil.rmtree(drive_adapter)
shutil.copytree(LOCAL_OUTPUT, drive_adapter)
print(f"Drive 저장 완료: {drive_adapter}")

# 파일 목록
for f in os.listdir(LOCAL_OUTPUT):
    size = os.path.getsize(os.path.join(LOCAL_OUTPUT, f))
    print(f"  {f}: {size / 1024**2:.1f} MB")

## 8. HF Hub 업로드

In [ ]:
api = HfApi()
api.upload_folder(
    folder_path=LOCAL_OUTPUT,
    path_in_repo="models/lora_adapter",
    repo_id=HF_REPO,
)
print(f"HF Hub 업로드 완료: {HF_REPO}/models/lora_adapter")

## 9. 추론 테스트

In [ ]:
model.eval()

test_questions = [
    "컴퓨터융합학부 졸업 요건이 어떻게 되나요?",
    "수강신청은 언제 하나요?",
    "오늘 학식 뭐 나와요?",
    "셔틀버스 시간표 알려주세요",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)